# Width sweeps on Kaggle

Runs a `width_sweep.py` sweep (see `docs/WIDTH-QUESTIONS.md`) as parallel shards on the
notebook's GPU(s).

**Settings (right panel):** Accelerator = GPU T4 x2 (recent PyTorch builds dropped P100 support), Internet = On.
Run it unattended with **Save Version → Save & Run All (Commit)**, then download
`width_results.zip` from the version's Output tab.

In [ ]:
SWEEP = "q3trim"   # a key of SWEEPS in width_sweep.py
SHARDS = 8         # parallel workers; the models are tiny, so several share one GPU
BRANCH = "master"

import os
os.environ["WIDTH_CACHE"] = "/tmp/width_cache"   # datasets cache, kept out of the output zip


In [ ]:
!git clone -q --depth 1 -b $BRANCH https://github.com/barsboldb/mpgnn.git
%cd mpgnn
!pip install -q torch-geometric
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git log -1 --format='%h %s'


In [ ]:
# Build every dataset the sweep needs in parallel (no-op for sweeps without cached data).
!python width_sweep.py $SWEEP --prepare


In [ ]:
import os, subprocess, time, torch

gpus = max(1, torch.cuda.device_count())
os.makedirs("results/width", exist_ok=True)
procs = []
for i in range(SHARDS):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i % gpus))
    log = open(f"results/width/{SWEEP}.shard{i}.log", "w")
    procs.append(subprocess.Popen(["python", "-u", "width_sweep.py", SWEEP, "--shard", f"{i}/{SHARDS}"],
                                  stdout=log, stderr=subprocess.STDOUT, env=env))
print(f"{SHARDS} shards on {gpus} GPU(s)")

t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    done = sum(1 for f in os.listdir("results/width") if f.startswith(f"{SWEEP}.shard") and f.endswith(".jsonl")
               for _ in open(f"results/width/{f}"))
    print(f"{(time.time() - t0) / 60:5.1f} min  runs saved: {done}", flush=True)
print("exit codes:", [p.returncode for p in procs])


In [ ]:
# Catch anything a shard missed or crashed on (no-op when complete), then analyze.
!python -u width_sweep.py $SWEEP
!python width_sweep.py $SWEEP --analyze
!python width_sweep.py $SWEEP --curves


In [ ]:
!cd results && zip -qr /kaggle/working/width_results.zip width && ls -lh /kaggle/working/width_results.zip
